# RetailMind AI — 01: Data Understanding & Quality Validation

**Purpose:** Determine whether the Amazon retail dataset is structurally valid and suitable for further analysis.

> This notebook covers **only** data quality and validation checks.  
> All exploratory charts are in `02_eda.ipynb`.


## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Libraries loaded successfully.")


Libraries loaded successfully.


## 2. Load Dataset

In [2]:
# Load the raw dataset — DO NOT modify this file
DATA_PATH = '../data/raw/Amazon.csv'

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")


Dataset loaded: 100,000 rows × 20 columns


## 3. Basic Dataset Audit

*(Previously completed — reproduced here for a self-contained record.)*

In [3]:
# --- Shape ---
print("=== Dataset Shape ===")
print(f"Rows   : {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

# --- Columns ---
print("\n=== Column Names ===")
print(df.columns.tolist())

# --- Data Types ---
print("\n=== Data Types ===")
print(df.dtypes)


=== Dataset Shape ===
Rows   : 100,000
Columns: 20

=== Column Names ===
['OrderID', 'OrderDate', 'CustomerID', 'CustomerName', 'ProductID', 'ProductName', 'Category', 'Brand', 'Quantity', 'UnitPrice', 'Discount', 'Tax', 'ShippingCost', 'TotalAmount', 'PaymentMethod', 'OrderStatus', 'City', 'State', 'Country', 'SellerID']

=== Data Types ===
OrderID              str
OrderDate            str
CustomerID           str
CustomerName         str
ProductID            str
ProductName          str
Category             str
Brand                str
Quantity           int64
UnitPrice        float64
Discount         float64
Tax              float64
ShippingCost     float64
TotalAmount      float64
PaymentMethod        str
OrderStatus          str
City                 str
State                str
Country              str
SellerID             str
dtype: object


In [4]:
# --- First 5 rows ---
print("=== First 5 Rows ===")
df.head()


=== First 5 Rows ===


,OrderID,OrderDate,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,Quantity,UnitPrice,Discount,Tax,ShippingCost,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
0,ORD0000001,2023-01-31,CUST001504,Vihaan Sharma,P00014,Drone Mini,Books,BrightLux,3,106.5900,0.0000,0.0000,0.0900,319.8600,Debit Card,Delivered,Washington,DC,India,SELL01967
1,ORD0000002,2023-12-30,CUST000178,Pooja Kumar,P00040,Microphone,Home & Kitchen,UrbanStyle,1,251.3700,0.0500,19.1000,1.7400,259.6400,Amazon Pay,Delivered,Fort Worth,TX,United States,SELL01298
2,ORD0000003,2022-05-10,CUST047516,Sneha Singh,P00044,Power Bank 20000mAh,Clothing,UrbanStyle,3,35.0300,0.1000,7.5700,5.9100,108.0600,Debit Card,Delivered,Austin,TX,United States,SELL00908
3,ORD0000004,2023-07-18,CUST030059,Vihaan Reddy,P00041,Webcam Full HD,Home & Kitchen,Zenith,5,33.5800,0.1500,11.4200,5.5300,159.6600,Cash on Delivery,Delivered,Charlotte,NC,India,SELL01164
4,ORD0000005,2023-02-04,CUST048677,Aditya Kapoor,P00029,T-Shirt,Clothing,KiddoFun,2,515.6400,0.2500,38.6700,9.2300,821.3600,Credit Card,Cancelled,San Antonio,TX,Canada,SELL01411


In [5]:
# --- Missing Values ---
print("=== Missing Values per Column ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

# --- Duplicate Rows ---
print(f"\n=== Duplicate Rows ===")
dup_rows = df.duplicated().sum()
print(f"Duplicate rows: {dup_rows}")


=== Missing Values per Column ===
No missing values found.

=== Duplicate Rows ===
Duplicate rows: 0


In [6]:
# --- OrderStatus Value Counts ---
print("=== OrderStatus Distribution ===")
print(df['OrderStatus'].value_counts())

# --- Unique counts for key categorical fields ---
print("\n=== Key Categorical Unique Counts ===")
for col in ['CustomerID', 'ProductID', 'Category', 'Brand', 'SellerID']:
    print(f"  Unique {col}: {df[col].nunique():,}")


=== OrderStatus Distribution ===
OrderStatus
Delivered    74628
Shipped      15192
Pending       4103
Returned      3049
Cancelled     3028
Name: count, dtype: int64

=== Key Categorical Unique Counts ===
  Unique CustomerID: 43,233
  Unique ProductID: 50
  Unique Category: 6
  Unique Brand: 10
  Unique SellerID: 1,999


In [7]:
# --- Numerical describe() for key fields ---
print("=== Numerical Summary (describe) ===")
df[['Quantity', 'UnitPrice', 'Discount', 'Tax', 'ShippingCost', 'TotalAmount']].describe()


=== Numerical Summary (describe) ===


,Quantity,UnitPrice,Discount,Tax,ShippingCost,TotalAmount
count,100000.0000,100000.0000,100000.0000,100000.0000,100000.0000,100000.0000
mean,3.0014,302.9057,0.0742,68.4689,7.4067,918.2565
std,1.4135,171.8408,0.0826,74.1312,4.3241,724.5083
min,1.0000,5.0000,0.0000,0.0000,0.0000,4.2700
25%,2.0000,154.1900,0.0000,15.9200,3.6800,340.8900
50%,3.0000,303.0700,0.0500,45.2500,7.3000,714.3150
75%,4.0000,451.5000,0.1000,96.0600,11.1500,1349.7650
max,5.0000,599.9900,0.3000,538.4600,15.0000,3534.9800


---
## 4. Order Date Validation

Convert `OrderDate` to datetime safely and check for invalid/missing dates.


In [8]:
# Convert OrderDate to datetime — 'coerce' turns unparseable values into NaT (not silent drop)
df['OrderDate_parsed'] = pd.to_datetime(df['OrderDate'], errors='coerce')

# Count valid vs invalid
valid_dates   = df['OrderDate_parsed'].notna().sum()
invalid_dates = df['OrderDate_parsed'].isna().sum()

print(f"Valid   dates : {valid_dates:,}")
print(f"Invalid dates : {invalid_dates:,}")

if invalid_dates > 0:
    print("\n⚠️  INVALID DATE RECORDS (showing first 10):")
    print(df[df['OrderDate_parsed'].isna()][['OrderID', 'OrderDate']].head(10))
else:
    print("\n✅ All OrderDate values parsed successfully — no invalid dates found.")

# Summary stats on valid dates
valid_df = df[df['OrderDate_parsed'].notna()]
print(f"\nEarliest valid date : {valid_df['OrderDate_parsed'].min().date()}")
print(f"Latest   valid date : {valid_df['OrderDate_parsed'].max().date()}")
print(f"Unique  dates       : {valid_df['OrderDate_parsed'].nunique():,}")


Valid   dates : 100,000
Invalid dates : 0

✅ All OrderDate values parsed successfully — no invalid dates found.

Earliest valid date : 2020-01-01
Latest   valid date : 2024-12-29
Unique  dates       : 1,825


---
## 5. Numerical Data Validation

For each key numerical column calculate min, max, mean, median, std, zero count, negative count, and missing count.


In [9]:
numerical_cols = ['Quantity', 'UnitPrice', 'Discount', 'Tax', 'ShippingCost', 'TotalAmount']

rows = []
for col in numerical_cols:
    series = df[col]
    rows.append({
        'Column'  : col,
        'Min'     : series.min(),
        'Max'     : series.max(),
        'Mean'    : round(series.mean(), 4),
        'Median'  : series.median(),
        'Std Dev' : round(series.std(), 4),
        'Zeros'   : (series == 0).sum(),
        'Negatives': (series < 0).sum(),
        'Missing' : series.isna().sum()
    })

num_summary = pd.DataFrame(rows).set_index('Column')
print("=== Numerical Data Validation Summary ===")
num_summary


=== Numerical Data Validation Summary ===


,Min,Max,Mean,Median,Std Dev,Zeros,Negatives,Missing
Column,,,,,,,,
Quantity,1.0000,5.0000,3.0014,3.0000,1.4135,0,0,0
UnitPrice,5.0000,599.9900,302.9057,303.0700,171.8408,0,0,0
Discount,0.0000,0.3000,0.0742,0.0500,0.0826,40246,0,0
Tax,0.0000,538.4600,68.4689,45.2500,74.1312,9878,0,0
ShippingCost,0.0000,15.0000,7.4067,7.3000,4.3241,20,0,0
TotalAmount,4.2700,3534.9800,918.2565,714.3150,724.5083,0,0,0


In [10]:
# Flag any column with negatives or unexpected zeros
print("=== Flags ===")
for col in numerical_cols:
    neg = (df[col] < 0).sum()
    zero = (df[col] == 0).sum()
    miss = df[col].isna().sum()
    flags = []
    if neg > 0:    flags.append(f"{neg} negatives")
    if col in ['Quantity', 'UnitPrice', 'TotalAmount'] and zero > 0:
        flags.append(f"{zero} zeros")
    if miss > 0:   flags.append(f"{miss} missing")
    status = "⚠️  " + ", ".join(flags) if flags else "✅ No issues"
    print(f"  {col:15s}: {status}")


=== Flags ===
  Quantity       : ✅ No issues
  UnitPrice      : ✅ No issues
  Discount       : ✅ No issues
  Tax            : ✅ No issues
  ShippingCost   : ✅ No issues
  TotalAmount    : ✅ No issues


---
## 6. Discount Validation

Inspect the actual `Discount` values to determine whether they represent the range **0–1** (fractional) or **0–100** (percentage points).


In [11]:
print("=== Discount Column Inspection ===")
print(f"Min    : {df['Discount'].min()}")
print(f"Max    : {df['Discount'].max()}")
print(f"Mean   : {df['Discount'].mean():.4f}")
print(f"Median : {df['Discount'].median():.4f}")
print(f"\nUnique values (sorted):")
print(sorted(df['Discount'].unique()))


=== Discount Column Inspection ===
Min    : 0.0
Max    : 0.3
Mean   : 0.0742
Median : 0.0500

Unique values (sorted):
[np.float64(0.0), np.float64(0.05), np.float64(0.1), np.float64(0.15), np.float64(0.2), np.float64(0.25), np.float64(0.3)]


In [12]:
# Sample 10 rows showing Discount alongside UnitPrice and TotalAmount
print("=== Sample Records: Discount vs Price vs Total ===")
df[['OrderID', 'UnitPrice', 'Discount', 'Tax', 'ShippingCost', 'TotalAmount']].sample(10, random_state=42)


=== Sample Records: Discount vs Price vs Total ===


,OrderID,UnitPrice,Discount,Tax,ShippingCost,TotalAmount
75721,ORD0075722,298.4500,0.0500,0.0000,10.4300,1144.5400
80184,ORD0080185,154.4800,0.0000,24.7200,7.6300,341.3100
19864,ORD0019865,400.3400,0.0500,91.2800,11.3300,1243.5800
76699,ORD0076700,175.5300,0.1000,7.9000,9.5400,175.4200
92991,ORD0092992,190.3700,0.0000,114.2200,11.1500,1077.2200
76434,ORD0076435,424.7000,0.0500,40.3500,3.7200,851.0000
84004,ORD0084005,559.8400,0.0500,95.7300,0.2700,627.8500
80917,ORD0080918,472.7600,0.0000,170.1900,4.7400,1120.4500
60767,ORD0060768,49.7500,0.0000,19.9000,0.6400,269.2900
50074,ORD0050075,143.0000,0.0500,32.6000,14.6100,454.7600


### Discount Interpretation

The observed `Discount` values range from **0.0 to 0.30** with unique values of `{0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30}`.

**Conclusion:** Discount is stored as a **fractional rate (0–1)**, not as a percentage (0–100).  
A value of `0.10` means a **10% discount**, and `0.30` means a **30% discount**.

The maximum discount applied is 30%. This is consistent with how discount is typically encoded in transactional datasets.

> ⚠️ **Note:** The raw `Discount` column has **not been modified**. Any transformation (e.g. converting to percentage) should be done in a processed copy during feature engineering.


---
## 7. ID Column Validation

Check `OrderID`, `CustomerID`, `ProductID`, and `SellerID` for missing values, uniqueness, and duplicates.


In [13]:
id_cols = ['OrderID', 'CustomerID', 'ProductID', 'SellerID']

id_rows = []
for col in id_cols:
    series = df[col]
    id_rows.append({
        'Column'     : col,
        'Total'      : len(series),
        'Missing'    : series.isna().sum(),
        'Unique'     : series.nunique(),
        'Duplicates' : series.duplicated().sum()
    })

id_summary = pd.DataFrame(id_rows).set_index('Column')
print("=== ID Column Validation Summary ===")
id_summary


=== ID Column Validation Summary ===


,Total,Missing,Unique,Duplicates
Column,,,,
OrderID,100000,0,100000,0
CustomerID,100000,0,43233,56767
ProductID,100000,0,50,99950
SellerID,100000,0,1999,98001


In [14]:
# --- Investigate duplicate OrderIDs specifically ---
dup_order_mask = df.duplicated(subset='OrderID', keep=False)
dup_count = dup_order_mask.sum()

print(f"Rows involved in duplicate OrderID: {dup_count}")

if dup_count > 0:
    print("\n⚠️  Duplicate OrderID records (showing first 20):")
    print(df[dup_order_mask].sort_values('OrderID').head(20))
else:
    print("\n✅ No duplicate OrderIDs found — each order record is unique.")


Rows involved in duplicate OrderID: 0

✅ No duplicate OrderIDs found — each order record is unique.


In [15]:
# Note on CustomerID and ProductID duplicates:
# CustomerID duplicates are EXPECTED — one customer can place multiple orders.
# ProductID duplicates are EXPECTED — the same product can appear in multiple orders.
# OrderID should be unique per order record.

cust_order_counts = df.groupby('CustomerID')['OrderID'].count()
print("=== Customer Order Frequency (CustomerID duplicate context) ===")
print(f"Customers with exactly 1 order: {(cust_order_counts == 1).sum():,}")
print(f"Customers with 2–5 orders     : {((cust_order_counts >= 2) & (cust_order_counts <= 5)).sum():,}")
print(f"Customers with 6–10 orders    : {((cust_order_counts >= 6) & (cust_order_counts <= 10)).sum():,}")
print(f"Customers with >10 orders     : {(cust_order_counts > 10).sum():,}")
print(f"Max orders by a single customer: {cust_order_counts.max()}")


=== Customer Order Frequency (CustomerID duplicate context) ===
Customers with exactly 1 order: 13,532
Customers with 2–5 orders     : 28,869
Customers with 6–10 orders    : 832
Customers with >10 orders     : 0
Max orders by a single customer: 10


---
## 8. Invalid / Suspicious Record Check

Systematic check of all known potential data quality issues. Records are **not deleted** — only counted and flagged.


In [16]:
total = len(df)

def pct(count, total=total):
    return f"{count / total * 100:.2f}%"

def status_flag(count, warn_threshold=0, investigate_threshold=100):
    if count == 0:
        return "PASS"
    elif count <= warn_threshold:
        return "WARNING"
    else:
        return "INVESTIGATE"

# Re-use parsed dates
invalid_date_count = df['OrderDate_parsed'].isna().sum()

checks = [
    {
        'Check'      : 'Quantity <= 0',
        'Count'      : int((df['Quantity'] <= 0).sum()),
        'Percentage' : pct((df['Quantity'] <= 0).sum()),
        'Status'     : status_flag((df['Quantity'] <= 0).sum())
    },
    {
        'Check'      : 'UnitPrice <= 0',
        'Count'      : int((df['UnitPrice'] <= 0).sum()),
        'Percentage' : pct((df['UnitPrice'] <= 0).sum()),
        'Status'     : status_flag((df['UnitPrice'] <= 0).sum())
    },
    {
        'Check'      : 'Discount < 0',
        'Count'      : int((df['Discount'] < 0).sum()),
        'Percentage' : pct((df['Discount'] < 0).sum()),
        'Status'     : status_flag((df['Discount'] < 0).sum())
    },
    {
        'Check'      : 'Discount > 1 (out of 0–1 range)',
        'Count'      : int((df['Discount'] > 1).sum()),
        'Percentage' : pct((df['Discount'] > 1).sum()),
        'Status'     : status_flag((df['Discount'] > 1).sum())
    },
    {
        'Check'      : 'Tax < 0',
        'Count'      : int((df['Tax'] < 0).sum()),
        'Percentage' : pct((df['Tax'] < 0).sum()),
        'Status'     : status_flag((df['Tax'] < 0).sum())
    },
    {
        'Check'      : 'ShippingCost < 0',
        'Count'      : int((df['ShippingCost'] < 0).sum()),
        'Percentage' : pct((df['ShippingCost'] < 0).sum()),
        'Status'     : status_flag((df['ShippingCost'] < 0).sum())
    },
    {
        'Check'      : 'TotalAmount <= 0',
        'Count'      : int((df['TotalAmount'] <= 0).sum()),
        'Percentage' : pct((df['TotalAmount'] <= 0).sum()),
        'Status'     : status_flag((df['TotalAmount'] <= 0).sum())
    },
    {
        'Check'      : 'Invalid OrderDate',
        'Count'      : int(invalid_date_count),
        'Percentage' : pct(invalid_date_count),
        'Status'     : status_flag(invalid_date_count)
    },
    {
        'Check'      : 'Missing OrderID',
        'Count'      : int(df['OrderID'].isna().sum()),
        'Percentage' : pct(df['OrderID'].isna().sum()),
        'Status'     : status_flag(df['OrderID'].isna().sum())
    },
    {
        'Check'      : 'Missing CustomerID',
        'Count'      : int(df['CustomerID'].isna().sum()),
        'Percentage' : pct(df['CustomerID'].isna().sum()),
        'Status'     : status_flag(df['CustomerID'].isna().sum())
    },
    {
        'Check'      : 'Missing ProductID',
        'Count'      : int(df['ProductID'].isna().sum()),
        'Percentage' : pct(df['ProductID'].isna().sum()),
        'Status'     : status_flag(df['ProductID'].isna().sum())
    },
    {
        'Check'      : 'Missing SellerID',
        'Count'      : int(df['SellerID'].isna().sum()),
        'Percentage' : pct(df['SellerID'].isna().sum()),
        'Status'     : status_flag(df['SellerID'].isna().sum())
    },
    {
        'Check'      : 'Duplicate rows (exact)',
        'Count'      : int(df.duplicated().sum()),
        'Percentage' : pct(df.duplicated().sum()),
        'Status'     : status_flag(df.duplicated().sum())
    },
    {
        'Check'      : 'Duplicate OrderID',
        'Count'      : int(df.duplicated(subset='OrderID').sum()),
        'Percentage' : pct(df.duplicated(subset='OrderID').sum()),
        'Status'     : status_flag(df.duplicated(subset='OrderID').sum())
    },
]

quality_report = pd.DataFrame(checks)
print("=== Data Quality Check Results ===")
quality_report


=== Data Quality Check Results ===


,Check,Count,Percentage,Status
0,Quantity <= 0,0,0.00%,PASS
1,UnitPrice <= 0,0,0.00%,PASS
2,Discount < 0,0,0.00%,PASS
3,Discount > 1 (out of 0–1 range),0,0.00%,PASS
4,Tax < 0,0,0.00%,PASS
5,ShippingCost < 0,0,0.00%,PASS
6,TotalAmount <= 0,0,0.00%,PASS
7,Invalid OrderDate,0,0.00%,PASS
8,Missing OrderID,0,0.00%,PASS
9,Missing CustomerID,0,0.00%,PASS


In [17]:
# Summary of statuses
status_counts = quality_report['Status'].value_counts()
print("=== Status Summary ===")
for status in ['PASS', 'WARNING', 'INVESTIGATE']:
    count = status_counts.get(status, 0)
    print(f"  {status:12s}: {count} check(s)")

total_issues = quality_report[quality_report['Status'] != 'PASS']['Count'].sum()
print(f"\nTotal flagged records (any issue): {total_issues:,}")


=== Status Summary ===
  PASS        : 14 check(s)
  INVESTIGATE : 0 check(s)

Total flagged records (any issue): 0


---
## 9. Final Data Quality Report


In [18]:
# Recompute all metrics for the final report
total_rows    = len(df)
total_cols    = df.shape[1]
missing_vals  = df.isnull().sum().sum()
dup_rows      = df.duplicated().sum()
invalid_dates = df['OrderDate_parsed'].isna().sum()
invalid_qty   = (df['Quantity'] <= 0).sum()
invalid_price = (df['UnitPrice'] <= 0).sum()
invalid_disc  = ((df['Discount'] < 0) | (df['Discount'] > 1)).sum()
invalid_tax   = (df['Tax'] < 0).sum()
invalid_ship  = (df['ShippingCost'] < 0).sum()
invalid_total = (df['TotalAmount'] <= 0).sum()
unique_custs  = df['CustomerID'].nunique()
unique_prods  = df['ProductID'].nunique()
unique_sell   = df['SellerID'].nunique()
date_min      = df['OrderDate_parsed'].min().date()
date_max      = df['OrderDate_parsed'].max().date()

report = {
    'Metric'                    : [
        'Total Rows',
        'Total Columns',
        'Total Missing Values (all columns)',
        'Duplicate Rows',
        'Invalid OrderDates',
        'Invalid Quantities (<=0)',
        'Invalid UnitPrices (<=0)',
        'Invalid Discounts (out of 0-1 range)',
        'Invalid Tax Values (<0)',
        'Invalid ShippingCosts (<0)',
        'Invalid TotalAmounts (<=0)',
        'Unique CustomerIDs',
        'Unique ProductIDs',
        'Unique SellerIDs',
        'Date Range Start',
        'Date Range End',
    ],
    'Value' : [
        f"{total_rows:,}",
        f"{total_cols}",
        f"{missing_vals:,}",
        f"{dup_rows:,}",
        f"{invalid_dates:,}",
        f"{invalid_qty:,}",
        f"{invalid_price:,}",
        f"{invalid_disc:,}",
        f"{invalid_tax:,}",
        f"{invalid_ship:,}",
        f"{invalid_total:,}",
        f"{unique_custs:,}",
        f"{unique_prods:,}",
        f"{unique_sell:,}",
        str(date_min),
        str(date_max),
    ]
}

report_df = pd.DataFrame(report).set_index('Metric')
print("=" * 55)
print("       FINAL DATA QUALITY REPORT — RetailMind AI")
print("=" * 55)
report_df


       FINAL DATA QUALITY REPORT — RetailMind AI


,Value
Metric,
Total Rows,"100,000"
Total Columns,21
Total Missing Values (all columns),0
Duplicate Rows,0
Invalid OrderDates,0
Invalid Quantities (<=0),0
Invalid UnitPrices (<=0),0
Invalid Discounts (out of 0-1 range),0
Invalid Tax Values (<0),0


---
## 10. Dataset Readiness Assessment


In [19]:
print("=" * 60)
print("        DATASET READINESS — RetailMind AI")
print("=" * 60)

# Re-evaluate all key flags
all_passed = all([
    invalid_dates  == 0,
    invalid_qty    == 0,
    invalid_price  == 0,
    invalid_disc   == 0,
    invalid_tax    == 0,
    invalid_ship   == 0,
    invalid_total  == 0,
    missing_vals   == 0,
    dup_rows       == 0,
])

if all_passed:
    print("\n✅  VERDICT: READY FOR EDA")
    print()
    print("Rationale:")
    print("  • No missing values across any of the 20 columns.")
    print("  • No duplicate rows or duplicate OrderIDs.")
    print("  • All 100,000 OrderDate values parse correctly.")
    print("  • No negative Quantity, UnitPrice, Tax, ShippingCost,")
    print("    or TotalAmount values detected.")
    print("  • Discount is consistently encoded as a fractional rate")
    print("    (0.0–0.30), with no out-of-range values.")
    print("  • All ID columns (OrderID, CustomerID, ProductID,")
    print("    SellerID) are fully populated.")
    print()
    print("  The dataset contains 100,000 clean, complete transaction")
    print("  records spanning 2020-01-01 to 2024-12-29, covering")
    print("  5 years of retail activity across multiple countries,")
    print("  categories, brands, and sellers.")
    print()
    print("  ➡  Proceed to: notebook/02_eda.ipynb")
else:
    print("\n⚠️  VERDICT: REQUIRES DATA CLEANING BEFORE EDA")
    print()
    print("Rationale:")
    if invalid_dates  > 0: print(f"  • {invalid_dates:,} invalid OrderDate values found.")
    if missing_vals   > 0: print(f"  • {missing_vals:,} missing values found across columns.")
    if dup_rows       > 0: print(f"  • {dup_rows:,} duplicate rows found.")
    if invalid_qty    > 0: print(f"  • {invalid_qty:,} records with Quantity <= 0.")
    if invalid_price  > 0: print(f"  • {invalid_price:,} records with UnitPrice <= 0.")
    if invalid_disc   > 0: print(f"  • {invalid_disc:,} records with invalid Discount values.")
    if invalid_tax    > 0: print(f"  • {invalid_tax:,} records with Tax < 0.")
    if invalid_ship   > 0: print(f"  • {invalid_ship:,} records with ShippingCost < 0.")
    if invalid_total  > 0: print(f"  • {invalid_total:,} records with TotalAmount <= 0.")
    print()
    print("  These issues should be resolved in a data cleaning notebook")
    print("  before proceeding to EDA.")


        DATASET READINESS — RetailMind AI

✅  VERDICT: READY FOR EDA

Rationale:
  • No missing values across any of the 20 columns.
  • No duplicate rows or duplicate OrderIDs.
  • All 100,000 OrderDate values parse correctly.
  • No negative Quantity, UnitPrice, Tax, ShippingCost,
    or TotalAmount values detected.
  • Discount is consistently encoded as a fractional rate
    (0.0–0.30), with no out-of-range values.
  • All ID columns (OrderID, CustomerID, ProductID,
    SellerID) are fully populated.

  The dataset contains 100,000 clean, complete transaction
  records spanning 2020-01-01 to 2024-12-29, covering
  5 years of retail activity across multiple countries,
  categories, brands, and sellers.

  ➡  Proceed to: notebook/02_eda.ipynb
